# From reports to market series

Inspect source fields, coverage and completeness on local data.

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
NB = HERE if (HERE / 'mandi_notes.py').is_file() else HERE / 'Notebook'
sys.path.insert(0, str(NB))
from mandi_notes import *

In [2]:
from Code.utils.data_audit import load_mandi_file
from Code.intelligence.states import state_id

sample = load_mandi_file("Onion.csv", mandi_dir=RAW, nrows=1000)
display(sample.isna().sum().loc[lambda x: x > 0].to_frame("Missing in first 1,000 raw rows"))
display(sample[["State Name", "District Name", "Market Name", "Variety", "Group", "Reported Date",
                "Modal Price (Rs./Quintal)", "Arrivals (Tonnes)"]].head(6))
snap = publication()
registry = pd.read_parquet(DAILY / "releases" / snap["release"]["release"] / "registry.parquet")
coverage = registry.assign(region=registry.state.map(state_id)).groupby("source").agg(
    crops=("crop", "nunique"), regions=("region", "nunique"), series=("series_id", "nunique"),
    first_report=("first_date", "min"), last_report=("last_date", "max"))
display(coverage)
print(f"Local commodity CSVs: {len(list(RAW.glob('*.csv')))}")

,"Missing in first 1,000 raw rows"


,State Name,District Name,Market Name,Variety,Group,Reported Date,Modal Price (Rs./Quintal),Arrivals (Tonnes)
0,Arunachal Pradesh,Tawang,Tawang,Other,Vegetables,09 Feb 2006,1500.0,0.9
1,Arunachal Pradesh,Tawang,Tawang,Other,Vegetables,07 Feb 2006,1800.0,0.9
2,Arunachal Pradesh,Tawang,Tawang,Other,Vegetables,24 Jan 2006,1800.0,0.7
3,Arunachal Pradesh,Tawang,Tawang,Other,Vegetables,19 Jan 2006,1800.0,0.9
4,Arunachal Pradesh,Tawang,Tawang,Other,Vegetables,13 Jan 2006,2000.0,1.0
5,Arunachal Pradesh,Tawang,Tawang,Other,Vegetables,09 Jan 2006,2000.0,0.7


,crops,regions,series,first_report,last_report
source,,,,,
agmarknet_archive,56,33,93242,2001-01-10,2024-02-02
datagov,19,31,25415,2024-02-03,2026-11-13


Local commodity CSVs: 325


In [3]:
def inspect(crop, region):
    sample = archive_sample(crop)
    view = sample if region == "All sampled regions" else sample[sample.state == region]
    if view.empty:
        print("No rows in this deterministic development sample.")
        return
    display(view.groupby("state", observed=True).agg(sample_rows=("price", "size"), markets=("market", "nunique"),
        reporting_days=("date", "nunique"))
        .sort_values("sample_rows", ascending=False).head(8))
    print(f"Sample dates: {view.date.min():%Y-%m-%d} to {view.date.max():%Y-%m-%d}")

controls(inspect, crop=CROPS, region=["All sampled regions", "Maharashtra", "Uttar Pradesh", "Punjab"])

,sample_rows,markets,reporting_days
state,,,
Uttar Pradesh,6287,243,3538
Punjab,4808,114,3150
Haryana,3132,104,2309
Odisha,2574,95,2005
Maharashtra,2136,109,1734
Kerala,1866,81,1366
West Bengal,1406,48,1184
Gujarat,1203,58,1077


Sample dates: 2001-04-18 to 2018-12-31


interactive(children=(Dropdown(description='Crop', options=('Onion', 'Potato', 'Wheat'), value='Onion'), Dropd…

The raw sample shows the archive's Group field. The recent feed carries a separate Grade field. Missing arrivals are counted as missing. The region table summarizes a deterministic sample of cleaned pre-2019 records.

**Takeaway** · Market, variety, grade, and source matter when defining comparable series. Reported modal price and a weekly median are different quantities.

**Next** · Compare prices and reporting patterns in Notebook 03.